# 06 - QA

Report-only checks (nothing halts): boundary coverage, unit sanity, GCM x scenario x
variable completeness, and the download manifest. Outputs land in `outputs/`.

In [ ]:
import sys
print("Python:", sys.executable)

import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr

# Pipeline root = climate/ (parent of notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.io import load_config, get_logger, append_manifest, sha256_file

cfg = load_config()
log = get_logger("06_qa")

RAW = ROOT / cfg["paths"]["raw"]
PROCESSED = ROOT / cfg["paths"]["processed"]
OUTPUTS = ROOT / cfg["paths"]["outputs"]
for p in (RAW, PROCESSED, OUTPUTS):
    p.mkdir(parents=True, exist_ok=True)

BBOX = cfg["study_area"]["bbox"]
log.info(f"bbox: lon {BBOX['lon_min']}..{BBOX['lon_max']}, lat {BBOX['lat_min']}..{BBOX['lat_max']}")

In [ ]:
import geopandas as gpd
import requests as _rq

qa = {}

# --- 1. Coverage: buffered boundary must sit inside the bbox (esp. the NV side) ---
bnd_json = _rq.get(f"{cfg['study_area']['boundary_url']}/query",
                   params={"where": "1=1", "outFields": "OBJECTID", "f": "geojson"},
                   timeout=120).json()
boundary = gpd.GeoDataFrame.from_features(bnd_json["features"], crs="EPSG:4326")
buf_ll = boundary.to_crs(epsg=cfg["crs"]["target_epsg"]) \
    .buffer(cfg["study_area"]["buffer_km"] * 1000)
buf_ll = gpd.GeoSeries(buf_ll, crs=f"EPSG:{cfg['crs']['target_epsg']}").to_crs("EPSG:4326")
w, s, e, n = buf_ll.total_bounds
inside = (w >= BBOX["lon_min"] and e <= BBOX["lon_max"]
          and s >= BBOX["lat_min"] and n <= BBOX["lat_max"])
qa["buffered_boundary_within_bbox"] = bool(inside)
if not inside:
    log.warning(f"buffered boundary {[round(x,3) for x in (w,s,e,n)]} exceeds bbox - "
                "NV/corridor cells would be clipped off; widen study_area.bbox")

# --- 2. Unit sanity on raw subsets ---
def check_units(path, var_hint):
    try:
        ds = xr.open_dataset(path)
        da = ds[list(ds.data_vars)[0]]
        units = str(da.attrs.get("units", "?"))
        sample = float(da.isel(time=slice(0, 30)).max()) if "time" in da.dims else float(da.max())
        return units, round(sample, 2)
    except Exception as e:
        return f"ERROR {e}", None

unit_rows = []
for sub in ["loca2", "gridmet", "caladapt"]:
    d = RAW / sub
    for p in sorted(d.glob("*.nc"))[:12] if d.exists() else []:
        u, samp = check_units(p, None)
        unit_rows.append({"file": p.name, "units": u, "sample_max": samp})
        if "pr" in p.name.split(".")[0] and ("kg" in u):
            log.info(f"{p.name}: native {u} - transform multiplies by 86400 to mm/day")
        if "tas" in p.name and u.strip() in ("K", "kelvin") :
            log.info(f"{p.name}: native Kelvin - transform converts to degC")
units_df = pd.DataFrame(unit_rows)
units_df.to_csv(OUTPUTS / "qa_units.csv", index=False)
qa["files_unit_checked"] = len(units_df)

# --- 3. Completeness matrix ---
L = cfg["sources"]["loca2"]
rows = []
loca2_dir = RAW / "loca2"
for gcm in L["gcms"]:
    member = L.get("member_overrides", {}).get(gcm, L["member"])
    for scen in ["historical"] + L["scenarios"]:
        for var in L["variables"]:
            n = len(list(loca2_dir.glob(f"{var}.{gcm}.{scen}.{member}.*__tahoe.nc"))) \
                if loca2_dir.exists() else 0
            rows.append({"gcm": gcm, "scenario": scen, "variable": var,
                         "files_present": n, "complete": n > 0})
comp = pd.DataFrame(rows)
comp.to_csv(OUTPUTS / "completeness_matrix.csv", index=False)
qa["loca2_cells_complete"] = f"{int(comp['complete'].sum())}/{len(comp)}"
missing = comp[~comp["complete"]]
if len(missing):
    log.warning(f"{len(missing)} GCM x scenario x variable cells missing - "
                "see completeness_matrix.csv")

# --- 4. Manifest ---
man_path = OUTPUTS / "manifest.csv"
if man_path.exists():
    man = pd.read_csv(man_path)
    qa["manifest_rows"] = len(man)
    qa["manifest_total_mb"] = round(man["size_bytes"].sum() / 1e6, 1)
else:
    qa["manifest_rows"] = 0
    log.warning("no manifest.csv yet - extract notebooks have not downloaded anything")

for k, v in qa.items():
    log.info(f"QA {k}: {v}")
pd.json_normalize(qa).to_csv(OUTPUTS / "qa_report.csv", index=False)
comp.pivot_table(index="gcm", columns=["scenario", "variable"],
                 values="files_present", aggfunc="sum") if len(comp) else comp